# P9 — Sensibilidad al tamaño de patch

## Objetivo
Determinar cómo cambia la discretización espacial al modificar `patch_size`.

Se consideran:

- 224;
- 256;
- 384;
- 512 px.

In [1]:

from pathlib import Path
import sys
import yaml

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró config.yaml. "
        "Ejecute Jupyter desde la carpeta del proyecto o notebooks/."
    )

ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(
    (ROOT / "config.yaml").read_text(encoding="utf-8")
)

print("Proyecto:", ROOT)


Proyecto: C:\Users\Nicolas Manco\Downloads\CONCRETE_DL


## 1. Cargar tabla de sensibilidad

In [ ]:
import pandas as pd, numpy as np
from PIL import Image, ImageOps
from src.data import axis_positions

OUT=ROOT/'results/audit'; OUT.mkdir(parents=True,exist_ok=True)
rows=[]
for ps in cfg['experiments']['patch_sizes']:
    for group in ['anomaly_field','test']:
        counts=[]
        for p in sorted((ROOT/'pictures'/group).glob('*')):
            if not p.is_file(): continue
            im=ImageOps.exif_transpose(Image.open(p)); w,h=im.size
            stride=max(1,round(ps*(1-cfg['field_inference']['overlap_fraction'])))
            n=len(axis_positions(w,ps,stride))*len(axis_positions(h,ps,stride))
            counts.append(n)
        rows.append({'patch_size':ps,'group':group,'images':len(counts),'total_patches':int(sum(counts)),
                     'mean_patches_per_image':float(np.mean(counts)),'median_patches_per_image':float(np.median(counts)),
                     'max_patches_per_image':int(max(counts))})
sensitivity=pd.DataFrame(rows)
sensitivity.to_csv(OUT/'patch_size_sensitivity.csv',index=False)
display(sensitivity)


## 2. Graficar cantidad de patches

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,5))
for group,part in sensitivity.groupby('group'):
    ax.plot(part['patch_size'],part['total_patches'],marker='o',label=group)
ax.set_xlabel('Patch size [px]'); ax.set_ylabel('Cantidad de patches'); ax.set_title('Sensibilidad al tamaño de patch')
ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig(ROOT/'results/audit/patch_count_sensitivity.png',dpi=220,bbox_inches='tight'); plt.show(); plt.close(fig)


## 3. Tabla de eficiencia espacial

In [ ]:
display(sensitivity[['patch_size','group','total_patches','median_patches_per_image','max_patches_per_image']])
print('P9 regeneró el CSV y la figura de sensibilidad a partir de las imágenes del proyecto.')


In [ ]:
from pathlib import Path
import hashlib
rows=[]
patterns={
    "P1":["audit/*notebook*","mosaics/*notebook*"],
    "P2":["audit/split_*","audit/cross_split_perceptual_similarity.csv","audit/photometric_shortcut_*.csv"],
    "P3":["training/baseline_*"],
    "P4":["training/resnet50_*"],
    "P5":["metrics/*"],
    "P6":["inference/field_*","inference/field_patches/*","heatmaps/anomaly_field__*","heatmaps/test__*"],
    "P7":["inference/inf_*","inference/inference_*"],
    "P8":["gradcam/*","heatmaps/inference__*","heatmaps/*probability_map*"],
    "P9":["audit/patch_size_sensitivity.csv","audit/patch_count_sensitivity.png"],
    "PAPER":["paper/*"]
}
for module,pats in patterns.items():
    seen=set()
    for pat in pats:
        for f in (ROOT/"results").glob(pat):
            if f.is_file() and f not in seen:
                seen.add(f)
                rows.append({
                    "module":module,
                    "file":str(f.relative_to(ROOT)),
                    "bytes":f.stat().st_size,
                    "sha256":hashlib.sha256(f.read_bytes()).hexdigest()
                })
trace=pd.DataFrame(rows).sort_values(["module","file"])
trace.to_csv(ROOT/"results/RESULTADOS_TRAZABILIDAD.csv",index=False)

print("P9 generó results/RESULTADOS_TRAZABILIDAD.csv y results/README.md")